In [10]:
import pandas as pd
from sklearn.preprocessing import minmax_scale
import os

In [11]:
df_future = pd.read_csv("../src/data/files/FUTURE_CAMPAIGN_PE_OFFERS.csv")

In [ ]:
#Filtering out offer-ids whose product's CODCUC is XXXXXXXXX
def filter_invalid_offers(df):
    """
    Filter out all rows where ID_OFERTA corresponds to any row having CODCUC as 'XXXXXXXXX'
    
    Parameters:
    df (pandas.DataFrame): DataFrame containing 'ID_OFERTA' and 'CODCUC' columns
    
    Returns:
    pandas.DataFrame: Filtered DataFrame with removed rows
    """
    # Get all ID_OFERTAs that have CODCUC as 'XXXXXXXXX'
    invalid_ids = df[df['CODCUC'] == 'XXXXXXXXX']['ID_OFERTA'].unique()
    
    # Filter out all rows with these ID_OFERTAs
    filtered_df = df[~df['ID_OFERTA'].isin(invalid_ids)]
    
    # Print some information about the filtering
    removed_count = len(df) - len(filtered_df)
    print(f"Removed {removed_count} rows")
    print(f"Found {len(invalid_ids)} unique ID_OFERTAs with CODCUC 'XXXXXXXXX'")
    
    return filtered_df

df_future_filtered = filter_invalid_offers(df_future)

Removed 11456 rows
Found 2011 unique ID_OFERTAs with CODCUC 'XXXXXXXXX'


In [15]:
df_future_filtered

,COD_PAIS,COD_PERIODO,COD_VENTA,COD_SAP,ID_OFERTA,COD_CATALOGO,ID_MACROESTRATEGIA,ID_TIPO_SUBESTRATEGIA,DES_TIPO_SUBESTRATEGIA,DES_TIPO_ESTRATEGIA,...,ES_GRATIS,ES_PADRE,ES_GRUPO_PADRE,FACTOR_CUADRE,FACTOR_REPETICION,COD_TIPO_OFERTA,COD_TIPO_PROFIT,ID_SUBESTRATEGIA,CODCUC,COMPOSITE_PRIMARY_KEY
0,PE,202502,103037,200106440,2728,45.0,7,993,993,OTRO,...,0,0,0,2,1.0,203,1.0,2582,P0210133000,993|2|1|0|0|P0210133000|
2,PE,202502,104805,210100498,2873,45.0,1085,993,993,OTRO,...,0,1,1,5,1.0,203,1.0,2727,P0216050000,993|2|1|0|1|P0216050000|
3,PE,202502,110175,200112287,3256,45.0,85,993,993,OTRO,...,0,0,0,2,1.0,203,1.0,3110,P0197145001,993|2|1|0|0|P0197145001|
6,PE,202502,18530,200092313,235,9.0,10,1,INDIVIDUAL,INDIVIDUAL,...,0,1,1,1,1.0,300,1.0,235,200012761,1|1|1|0|1|200012761|
7,PE,202501,117565,210104392,3250,45.0,1414,1,INDIVIDUAL,INDIVIDUAL,...,0,1,1,1,1.0,203,1.0,2984,P0132050000,1|2|1|0|1|P0132050000|
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46885,PE,202501,112286,210100495,4894,45.0,1442,993,993,OTRO,...,0,0,0,5,1.0,203,1.0,4628,P0216050000,993|2|1|0|0|P0216050000|
46886,PE,202502,112103,200118067,3387,45.0,1057,993,993,OTRO,...,0,1,1,5,1.0,203,1.0,3241,P0132061007,993|2|1|0|1|P0132061007|
46887,PE,202502,110005,200115338,3242,45.0,1141,993,993,OTRO,...,0,1,1,8,1.0,203,1.0,3096,P0408062000,993|2|1|0|1|P0408062000|
46888,PE,202501,103166,200106564,3971,45.0,1460,993,993,OTRO,...,0,0,0,1,1.0,203,1.0,3705,P0197101008,993|2|1|0|0|P0197101008|


In [15]:
# For future campaigns
#Filtering out product's whose CODCUC is XXXXXXXXX
df_future_filtered = df_future[df_future["CODCUC"] != "XXXXXXXXX"]

print(df_future_filtered.shape)

(44125, 26)


In [16]:
# For future campaigns
# Dropping the COMPOSITE_PRIMARY_KEY column 
df_future_filtered = df_future_filtered.drop("COMPOSITE_PRIMARY_KEY", axis=1)  # Remove "Column2"
print(df_future_filtered.shape)

(44125, 25)


In [17]:
# Finding the number of campaigns and the products present in that campaign
unique_occurrences = df_future_filtered["COD_PERIODO"].value_counts()

print("Unique occurrences:")
print(unique_occurrences)

Unique occurrences:
COD_PERIODO
202501    24918
202502    16509
202503     2698
Name: count, dtype: int64


In [18]:
# Creation of Composite key by combining the 6 attributes
df_future_filtered["Composite_key"] = df_future_filtered[["DES_TIPO_SUBESTRATEGIA","DES_TIPO_GRUPO","CODCUC","ES_PADRE","ES_GRATIS","FACTOR_REPETICION"]].astype(str).agg('|'.join, axis=1)

In [19]:
# Divide the DataFrame based on unique values in the 'COD_PERIODO' column
df_futures = {campaign_id: df_subset for campaign_id, df_subset in df_future_filtered.groupby('COD_PERIODO')}

In [20]:
# Assuming dfs contains the smaller DataFrames (subsets) from the previous steps
# Example of transformations on each sub_df
for i, (campaign_id, sub_df) in enumerate(df_futures.items(), 1):
    
    # Perform the groupby and aggregation
    sub_df_concatenated = sub_df.groupby("ID_OFERTA")[["Composite_key"]].agg(
        {'Composite_key': lambda x: '|'.join(x)}
    ).reset_index()
    
    # Save the result to a CSV file with a dynamic name
    file_name = f"{campaign_id}.csv"
    sub_df_concatenated.to_csv(file_name, index=False)
    
    print(f"Saved {file_name}")

Saved 202501.csv
Saved 202502.csv
Saved 202503.csv


In [22]:
# Calculate the percentage of offers present in the next campaign
def calculate_key_overlap_percentage(csv1_path, csv2_path):

    df1 = pd.read_csv(csv1_path)
    df2 = pd.read_csv(csv2_path)
    
    # Clean the composite keys (remove whitespace and convert to lowercase)
    # df1['Composite_key'] = df1['Composite_key'].str.strip().str.lower()
    # df2['Composite_key'] = df2['Composite_key'].str.strip().str.lower()
    
    # Get unique composite keys from both DataFrames
    keys_in_csv1 = df1['Composite_key']
    keys_in_csv2 = df2['Composite_key']
    unique_keys_in_csv1 = set(df1['Composite_key'].unique())
    unique_keys_in_csv2 = set(df2['Composite_key'].unique())
    
    # Find overlapping keys
    common_keys = unique_keys_in_csv1.intersection(unique_keys_in_csv2)
    
    # Calculate percentage
    overlap_percentage = (len(common_keys) / len(unique_keys_in_csv1)) * 100
    
    # Compile statistics
    stats = {
        'keys_in_csv1': len(keys_in_csv1),
        'keys_in_csv2': len(keys_in_csv2),
        'total_keys_csv1': len(unique_keys_in_csv1),
        'total_keys_csv2': len(unique_keys_in_csv2),
        'common_keys': len(common_keys),
        'overlap_percentage': round(overlap_percentage, 2)
    }
    
    return overlap_percentage, stats



In [23]:
import os
import pandas as pd
csv1_path = "202501.csv"
csv2_path = "202502.csv"

csv1_value = os.path.splitext(os.path.basename(csv1_path))[0]
csv2_value = os.path.splitext(os.path.basename(csv2_path))[0]
percentage, stats = calculate_key_overlap_percentage(csv1_path, csv2_path)

print(f"\nResults:")
print(f"Total unique offers in {csv1_value}: {stats['keys_in_csv1']}")
print(f"Total unique offers in {csv1_value}: {stats['total_keys_csv1']}")
print(f"Total unique offers in {csv1_value}: {stats['keys_in_csv2']}")
print(f"Total unique offers in {csv2_value}: {stats['total_keys_csv2']}")
print(f"Number of common offers: {stats['common_keys']}")
print(f"Percentage of offers of {csv1_value} present in {csv2_value}: {stats['overlap_percentage']}%")


Results:
Total unique offers in 202501: 5411
Total unique offers in 202501: 3907
Total unique offers in 202501: 3727
Total unique offers in 202502: 2801
Number of common offers: 1064
Percentage of offers of 202501 present in 202502: 27.23%


In [ ]:
Results:
Total unique offers in 202501: 3238
Total unique offers in 202502: 1992
Number of common offers: 537
Percentage of offers of 202501 present in 202502: 16.58%

In [25]:
csv1_path = "202502.csv"
csv2_path = "202503.csv"

csv1_value = os.path.splitext(os.path.basename(csv1_path))[0]
csv2_value = os.path.splitext(os.path.basename(csv2_path))[0]
percentage, stats = calculate_key_overlap_percentage(csv1_path, csv2_path)

print(f"\nResults:")
print(f"Total unique offers in {csv1_value}: {stats['keys_in_csv1']}")
print(f"Total unique offers in {csv1_value}: {stats['total_keys_csv1']}")
print(f"Total unique offers in {csv1_value}: {stats['keys_in_csv2']}")
print(f"Total unique offers in {csv2_value}: {stats['total_keys_csv2']}")
print(f"Number of common offers: {stats['common_keys']}")
print(f"Percentage of offers of {csv1_value} present in {csv2_value}: {stats['overlap_percentage']}%")


Results:
Total unique offers in 202502: 3727
Total unique offers in 202502: 2801
Total unique offers in 202502: 1422
Total unique offers in 202503: 1031
Number of common offers: 560
Percentage of offers of 202502 present in 202503: 19.99%
